# Notebook 04 — QLoRA Fine-Tuning (SFT)

**Model:** `Qwen/Qwen2.5-3B-Instruct` (fallback: `google/gemma-2-2b-it`)  
**Method:** 4-bit QLoRA via PEFT + TRL SFTTrainer  
**GPU:** T4 (15 GB VRAM)

**Steps:**
1. Mount Drive & install packages
2. Load & format `qa_train.jsonl` into ChatML
3. Load base model in 4-bit quantization
4. Configure LoRA
5. Train with SFTTrainer
6. Save adapter to Drive + push to HuggingFace Hub

> **Prerequisites:** Notebooks 01 and 02 must be complete.

In [2]:
from pathlib import Path
import os

# Tự động tìm đường dẫn
if os.path.exists('/content/'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/NLP_Final/Source/')
else:
    BASE = "../"

print(f'Base directory: {BASE}')
print(os.listdir(BASE))

Mounted at /content/drive
Base directory: /content/drive/MyDrive/NLP_Final/Source
['.gitignore', 'requirements.txt', 'scripts', '.git', 'data', 'notebooks', 'models', 'results', '.env']


In [3]:
!pip install -q transformers==4.46.3
!pip install -q peft==0.13.2
!pip install -q trl==0.12.1
!pip install -q accelerate==1.1.0
!pip install -q datasets==3.2.0
!pip install -q huggingface-hub==0.26.5
!pip install -q bitsandbytes==0.45.0
!pip install -q torch>=2.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 504.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.9/310.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━

In [4]:
import importlib
print("=== Installed versions ===")
for pkg in ['transformers', 'peft', 'trl', 'accelerate', 'datasets', 'bitsandbytes', 'torch']:
    try:
        print(f"{pkg}: {importlib.metadata.version(pkg)}")
    except:
        print(f"{pkg}: not found")

=== Installed versions ===
transformers: 4.46.3
peft: 0.13.2
trl: 0.12.1
accelerate: 1.1.0
datasets: 3.2.0
bitsandbytes: 0.45.0
torch: 2.10.0+cpu


## 1. Verify GPU

In [1]:
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {total:.1f} GB')
    COMPUTE_DTYPE = torch.bfloat16 if total > 30 else torch.float16
    print(f'Using compute dtype: {COMPUTE_DTYPE}')
else:
    raise RuntimeError('GPU not available. Switch runtime to GPU in Colab.')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB
Using compute dtype: torch.float16


## 2. Load and Format Training Data

In [ ]:

import json, os

SYSTEM_PROMPT = (
    'Bạn là trợ lý AI của Trường Đại học Tôn Đức Thắng (TDTU). '
    'Bạn trả lời các câu hỏi về quy chế, quy định, chính sách của trường một cách chính xác và đầy đủ. '
    'Trả lời bằng tiếng Việt. Nếu không có đủ thông tin, hãy nói rõ điều đó.'
)

def format_chatml(pair: dict) -> str:
    """Single-turn QA pair → Qwen2.5 ChatML string."""
    return (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{pair["question"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{pair["answer"]}<|im_end|>'
    )

def format_chatml_multi_turn(conv: dict) -> str:
    """Multi-turn conversation → Qwen2.5 ChatML string.
    DataCollatorForCompletionOnlyLM will mask all turns except assistant ones.
    """
    text = f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
    for turn in conv['turns']:
        text += f'<|im_start|>{turn["role"]}\n{turn["content"]}<|im_end|>\n'
    return text.rstrip('\n')

# Load QA pairs (single-turn)
train_pairs, test_pairs = [], []
with open(f'{BASE}/data/qa_filtered/qa_train.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        train_pairs.append(json.loads(line.strip()))
with open(f'{BASE}/data/qa_filtered/qa_test.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        test_pairs.append(json.loads(line.strip()))

# Only train on QA pairs approved via review_ui
verified_train = [p for p in train_pairs if p.get("human_verified", False)]
if verified_train:
    train_texts = [format_chatml(p) for p in verified_train]
    print(f'Human-verified QA pairs: {len(verified_train)} / {len(train_pairs)} total')
else:
    train_texts = [format_chatml(p) for p in train_pairs]
    print(f'WARNING: 0 human-verified pairs — using all {len(train_pairs)} pairs')
    print('  → Run review_ui to approve pairs before retraining.')
# Use all human-verified test pairs for eval; fallback to full test set
verified_test = [p for p in test_pairs if p.get('human_verified', False)]
eval_pool     = verified_test if verified_test else test_pairs
eval_texts    = [format_chatml(p) for p in eval_pool]

# Load conversations (multi-turn) — optional, graceful fallback
conv_texts = []
CONV_PATH  = f'{BASE}/data/qa_filtered/conversations_train.jsonl'
if os.path.exists(CONV_PATH):
    with open(CONV_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                conv = json.loads(line)
                # Only include conversations with valid turns
                if conv.get('turns') and len(conv['turns']) >= 4:
                    conv_texts.append(format_chatml_multi_turn(conv))
    print(f'Loaded {len(conv_texts)} multi-turn conversations')
else:
    print('conversations_train.jsonl not found — training on QA pairs only.')

# Combine
all_train_texts = train_texts + conv_texts

lengths = [len(t) for t in all_train_texts]
print(f'\nDataset composition:')
print(f'  Single-turn QA pairs : {len(train_texts)}')
print(f'  Multi-turn convs     : {len(conv_texts)}')
print(f'  Total train samples  : {len(all_train_texts)}')
print(f'  Eval samples         : {len(eval_texts)}')
print(f'\nChar length — avg: {sum(lengths)/len(lengths):.0f}  max: {max(lengths)}')
print(f'\nSample single-turn:\n{train_texts[0][:300]}…')
if conv_texts:
    print(f'\nSample multi-turn:\n{conv_texts[0][:500]}…')


## 3. Create HuggingFace Dataset

In [ ]:

from datasets import Dataset

train_dataset = Dataset.from_dict({'text': all_train_texts})
eval_dataset  = Dataset.from_dict({'text': eval_texts})

print(f'Train dataset: {train_dataset}')
print(f'Eval dataset : {eval_dataset}')


Train dataset: Dataset({
    features: ['text'],
    num_rows: 1309
})
Eval dataset : Dataset({
    features: ['text'],
    num_rows: 264
})


## 4. Load Base Model with 4-bit Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
# Fallback: MODEL_ID = 'google/gemma-2-2b-it'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

print(f'Loading base model: {MODEL_ID}')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Model loaded. Parameters: {model.num_parameters()/1e9:.2f}B')
print(f'Trainable parameters before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M')

Loading base model: Qwen/Qwen2.5-3B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded. Parameters: 3.09B
Trainable parameters before LoRA: 311.3M


## 5. Configure QLoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()  # required for gradient_checkpointing + PEFT

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M params ({100*trainable/total:.2f}%)')

Trainable: 29.93M / 1728.61M params (1.73%)


## 6. Training Configuration

In [ ]:
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

SFT_OUTPUT = f'{BASE}/models/sft_checkpoint'

training_args = SFTConfig(
    output_dir=SFT_OUTPUT,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,    # effective batch = 16, safer on T4
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    optim='paged_adamw_32bit',
    fp16=(COMPUTE_DTYPE == torch.float16),
    bf16=(COMPUTE_DTYPE == torch.bfloat16),
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    max_grad_norm=0.3,
    dataloader_num_workers=0,          # Required for Colab
    report_to='none',
    gradient_checkpointing=True,       # Saves VRAM at cost of ~30% speed
    # SFTConfig-specific
    max_seq_length=1024,               # fits multi-turn conversations (4-8 turns)
    dataset_text_field='text',
    packing=False,
)

# Only compute loss on assistant turns
response_template = tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

print('Training args configured.')

Training args configured.


## 7. Initialize SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

print('SFTTrainer initialized.')
print('max_seq_length=1024 — fits multi-turn conversations on T4 with gradient_checkpointing=True')

# Check VRAM before training
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved  = torch.cuda.memory_reserved(0) / 1e9
    print(f'VRAM allocated: {allocated:.2f} GB / reserved: {reserved:.2f} GB')


Map:   0%|          | 0/1309 [00:00<?, ? examples/s]

Map:   0%|          | 0/264 [00:00<?, ? examples/s]

SFTTrainer initialized.
max_seq_length=1024 — fits multi-turn conversations on T4 with gradient_checkpointing=True
VRAM allocated: 2.80 GB / reserved: 3.58 GB


## 8. Train

In [ ]:
print('Starting training...')

train_result = trainer.train()

print('\n=== Training Complete ===')
print(f'Train loss: {train_result.training_loss:.4f}')
print(f'Runtime: {train_result.metrics["train_runtime"]:.0f}s')
print(f'Samples/sec: {train_result.metrics["train_samples_per_second"]:.2f}')

Starting training...


Epoch,Training Loss,Validation Loss
0,1.420900,1.398126
1,1.103500,1.320767
2,0.920000,1.356136


Could not locate the best model at /content/drive/MyDrive/NLP_Final/Source/models/sft_checkpoint/checkpoint-163/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.



=== Training Complete ===
Train loss: 1.1903
Runtime: 2516s
Samples/sec: 1.56


## 9. Save Adapter to Drive

In [ ]:
trainer.save_model(SFT_OUTPUT)
tokenizer.save_pretrained(SFT_OUTPUT)

print(f'Adapter saved to {SFT_OUTPUT}')

import os
saved_files = os.listdir(SFT_OUTPUT)
print(f'Files: {saved_files}')

Adapter saved to /content/drive/MyDrive/NLP_Final/Source/models/sft_checkpoint
Files: ['checkpoint-243', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'training_args.bin']


## 10. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv(os.path.join(BASE, '.env'))

HF_TOKEN = os.environ.get("HF_TOKEN")
HF_USERNAME = os.environ.get("HF_USERNAME")
REPO_NAME = 'tdtu-student-regulations-qa'

if HF_TOKEN and HF_USERNAME:
    login(token=HF_TOKEN)
    trainer.model.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}')
    tokenizer.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}')
    print(f'Pushed to: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}')
else:
    print('Set HF_TOKEN and HF_USERNAME to push to HuggingFace Hub.')

## 11. Quick Inference Test

In [ ]:
from peft import PeftModel

def generate_answer(question: str, model, tokenizer,
                    context_chunks: list[str] = None,
                    max_new_tokens: int = 256) -> str:
    if context_chunks:
        context_str = '\n\n'.join([f'[{i+1}] {c}' for i, c in enumerate(context_chunks)])
        user_content = (
            f'Dựa vào các đoạn văn bản sau từ quy chế TDTU:\n\n'
            f'{context_str}\n\nCâu hỏi: {question}'
        )
    else:
        user_content = question

    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{user_content}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids('<|im_end|>'),
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# Test with a sample question
model.eval()
test_q = test_pairs[5]['question'] if test_pairs else 'Điều kiện xét học bổng là gì?'
answer = generate_answer(test_q, model, tokenizer)

print(f'Q: {test_q}')
print(f'A: {answer}')

Q: Nếu em gặp vấn đề bất thường trên mạng liên quan đến trường, em nên làm gì?
A: Dạ vâng! Nếu có sự cố bất thường nào đó liên quan đến trường, em cần báo ngay cho phòng chức năng hoặc khoa của mình để được giải quyết kịp thời. Không tự ý xử lý nhé, vì có thể ảnh hưởng đến quyền lợi của trường.


---
**Next step:** Run `05_experiments_eval.ipynb` to compare all 4 configurations and compute evaluation metrics.